In [3]:
!pip -q install spafe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 3.4 MB/s eta 0:00:00


In [6]:
from __future__ import annotations

import argparse
import sys
import zipfile
import subprocess
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Optional, Callable, Dict, Any

import numpy as np
import requests
from tqdm import tqdm
import librosa

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix


# ----------------------------
# Helpers: detect notebook + install
# ----------------------------

def in_notebook() -> bool:
    try:
        from IPython import get_ipython  # type: ignore
        ip = get_ipython()
        if ip is None:
            return False
        return "IPKernelApp" in ip.config
    except Exception:
        return False


def ensure_spafe_installed():
    try:
        import spafe  # noqa: F401
        return
    except ModuleNotFoundError:
        if not in_notebook():
            raise ImportError(
                "Missing package 'spafe' (required for PLP).\n"
                "Install it with: pip install spafe"
            )
        print("spafe not found. Installing now (Colab/Jupyter)...")
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "spafe"])
        import spafe  # noqa: F401
        print("spafe installed.")


# ----------------------------
# Data utilities (FSDD)
# ----------------------------

FSDD_ZIP_URL = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/heads/master.zip"


def download_and_prepare_fsdd(root_dir: Path) -> Path:
    """
    Download and extract FSDD from the official GitHub repo zip (if needed).
    Returns: root_dir/recordings containing wav files.
    """
    root_dir.mkdir(parents=True, exist_ok=True)
    recordings_dir = root_dir / "recordings"

    if recordings_dir.exists() and any(recordings_dir.glob("*.wav")):
        return recordings_dir

    zip_path = root_dir / "fsdd_master.zip"
    if not zip_path.exists():
        print(f"Downloading FSDD zip to: {zip_path}")
        r = requests.get(FSDD_ZIP_URL, stream=True, timeout=120)
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"Extracting: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(root_dir)

    extracted_recordings = root_dir / "free-spoken-digit-dataset-master" / "recordings"
    if not extracted_recordings.exists():
        raise FileNotFoundError("Could not find extracted recordings directory.")

    recordings_dir.mkdir(exist_ok=True)
    for wav in extracted_recordings.glob("*.wav"):
        target = recordings_dir / wav.name
        if not target.exists():
            target.write_bytes(wav.read_bytes())

    return recordings_dir


def parse_fsdd_label(wav_path: Path) -> int:
    """
    FSDD filenames: {digit}_{speaker}_{index}.wav
    Example: 7_jackson_32.wav
    """
    digit_str = wav_path.stem.split("_", 1)[0]
    return int(digit_str)


def list_dataset(recordings_dir: Path) -> List[Tuple[Path, int]]:
    wavs = sorted(recordings_dir.glob("*.wav"))
    if not wavs:
        raise FileNotFoundError(f"No wav files found in {recordings_dir}")
    return [(p, parse_fsdd_label(p)) for p in wavs]


# ----------------------------
# Noise + feature extraction
# ----------------------------

def add_awgn_at_snr(y: np.ndarray, snr_db: float, rng: np.random.Generator) -> np.ndarray:
    """
    Add white Gaussian noise to y at target SNR (dB).
    """
    y = y.astype(np.float32, copy=False)
    sig_power = float(np.mean(y ** 2) + 1e-12)
    snr_linear = 10 ** (snr_db / 10.0)
    noise_power = sig_power / snr_linear

    noise = rng.normal(0.0, np.sqrt(noise_power), size=y.shape).astype(np.float32)
    y_noisy = y + noise
    return np.clip(y_noisy, -1.0, 1.0)


def summarize_frames_fixed(feat: np.ndarray, target_dim: int) -> np.ndarray:
    """
    Turn frame features into a fixed-length vector of size 2*target_dim:
      [mean over time for each coeff] + [std over time for each coeff]

    Handles these issues robustly:
    - feat could be (n_coeffs, n_frames) OR (n_frames, n_coeffs)
    - very short utterances where frames < coeffs (transpose heuristic would break)
    - feat might have n_coeffs != target_dim (pad/truncate to target_dim)
    """
    feat = np.asarray(feat, dtype=np.float32)

    # Handle weird outputs
    if feat.ndim == 1:
        # treat as a single frame
        feat = feat.reshape(-1, 1)

    if feat.ndim != 2:
        raise ValueError(f"Expected 2D feature array, got shape {feat.shape}")

    # Decide orientation using target_dim if possible
    if feat.shape[0] == target_dim:
        f = feat  # (target_dim, n_frames)
    elif feat.shape[1] == target_dim:
        f = feat.T  # (target_dim, n_frames)
    else:
        # fallback: assume smaller axis is coeffs
        f = feat if feat.shape[0] < feat.shape[1] else feat.T

    # Now f is (n_coeffs, n_frames) but n_coeffs might not equal target_dim
    n_coeffs, n_frames = f.shape

    # Edge case: no frames
    if n_frames == 0:
        return np.zeros((2 * target_dim,), dtype=np.float32)

    # Pad or truncate coefficient dimension
    if n_coeffs < target_dim:
        pad = np.zeros((target_dim - n_coeffs, n_frames), dtype=np.float32)
        f = np.vstack([f, pad])
    elif n_coeffs > target_dim:
        f = f[:target_dim, :]

    mu = np.mean(f, axis=1)
    sd = np.std(f, axis=1)
    return np.concatenate([mu, sd], axis=0).astype(np.float32)


def extract_mfcc_vector(
    wav_path: Path,
    sr: int,
    snr_db: Optional[float],
    rng: np.random.Generator,
    n_mfcc: int = 13,
    win_len_s: float = 0.025,
    hop_len_s: float = 0.010,
    n_fft: int = 512,
) -> np.ndarray:
    y, _ = librosa.load(str(wav_path), sr=sr, mono=True)
    if snr_db is not None:
        y = add_awgn_at_snr(y, snr_db, rng)

    win_length = int(round(win_len_s * sr))
    hop_length = int(round(hop_len_s * sr))

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
    )  # usually (n_mfcc, n_frames)

    return summarize_frames_fixed(mfcc, target_dim=n_mfcc)


# ----------------------------
# PLP extraction (signature-robust)
# ----------------------------

def _pick_param_name(sig: inspect.Signature, candidates: List[str]) -> Optional[str]:
    names = list(sig.parameters.keys())

    for c in candidates:
        if c in sig.parameters:
            return c

    for n in names:
        low = n.lower()
        for c in candidates:
            if c in low:
                return n
    return None


def _call_spafe_plp_robust(
    spafe_plp_func,
    y: np.ndarray,
    sr: int,
    n_ceps: int,
    win_len_s: float,
    hop_len_s: float,
):
    sig = inspect.signature(spafe_plp_func)
    params = list(sig.parameters.values())

    # Build positional base args: always y, and often sr/fs as 2nd arg
    base_args = [y]
    if len(params) >= 2:
        p1 = params[1].name.lower()
        if p1 in {"fs", "sr", "sample_rate", "sampling_rate"} or params[1].default is inspect._empty:
            base_args = [y, sr]

    cep_name = _pick_param_name(sig, ["num_ceps", "n_ceps", "nceps", "order", "ncep", "cep"])
    win_name = _pick_param_name(sig, ["win_len", "winlen", "win_length", "window_length", "frame_len", "frame_length"])
    hop_name = _pick_param_name(sig, ["win_hop", "hop_len", "hop_length", "hop", "step", "win_step", "frame_hop", "frame_step"])
    pre_name = _pick_param_name(sig, ["pre_emph", "preemph", "pre_emphasis"])
    norm_name = _pick_param_name(sig, ["normalize", "norm"])

    def kw_if_allowed(name: Optional[str], value) -> Dict[str, Any]:
        if not name:
            return {}
        p = sig.parameters[name]
        if p.kind == inspect.Parameter.POSITIONAL_ONLY:
            return {}
        return {name: value}

    kwargs_best: Dict[str, Any] = {}
    kwargs_best.update(kw_if_allowed(cep_name, n_ceps))
    kwargs_best.update(kw_if_allowed(win_name, win_len_s))
    kwargs_best.update(kw_if_allowed(hop_name, hop_len_s))
    if pre_name:
        kwargs_best.update(kw_if_allowed(pre_name, 1))
    if norm_name:
        kwargs_best.update(kw_if_allowed(norm_name, 0))

    trials: List[Tuple[List[Any], Dict[str, Any]]] = []

    # If cep is positional-only, try passing as third positional
    if cep_name and sig.parameters[cep_name].kind == inspect.Parameter.POSITIONAL_ONLY:
        if len(params) >= 3:
            trials.append((base_args + [n_ceps], {}))

    trials.append((base_args, kwargs_best))

    kwargs_no_win = dict(kwargs_best)
    if win_name:
        kwargs_no_win.pop(win_name, None)
    if hop_name:
        kwargs_no_win.pop(hop_name, None)
    trials.append((base_args, kwargs_no_win))

    kwargs_min = dict(kwargs_no_win)
    if pre_name:
        kwargs_min.pop(pre_name, None)
    if norm_name:
        kwargs_min.pop(norm_name, None)
    trials.append((base_args, kwargs_min))

    trials.append((base_args, {}))

    last_err: Optional[Exception] = None
    for a, k in trials:
        try:
            return spafe_plp_func(*a, **k)
        except TypeError as e:
            last_err = e
            continue

    raise TypeError(
        "Could not call spafe PLP with your installed spafe version.\n"
        f"Signature seen: {sig}\n"
        f"Last error: {last_err}"
    )


def extract_plp_vector(
    wav_path: Path,
    sr: int,
    snr_db: Optional[float],
    rng: np.random.Generator,
    n_ceps: int = 13,
    win_len_s: float = 0.025,
    hop_len_s: float = 0.010,
) -> np.ndarray:
    ensure_spafe_installed()

    spafe_plp_func = None
    try:
        from spafe.features.rplp import plp as spafe_plp_func  # type: ignore
    except Exception:
        try:
            from spafe.features.plp import plp as spafe_plp_func  # type: ignore
        except Exception as e:
            raise ImportError("Could not import PLP from spafe (tried rplp.plp and plp.plp).") from e

    y, _ = librosa.load(str(wav_path), sr=sr, mono=True)
    if snr_db is not None:
        y = add_awgn_at_snr(y, snr_db, rng)

    plp_feat = _call_spafe_plp_robust(
        spafe_plp_func,
        y=y,
        sr=sr,
        n_ceps=n_ceps,
        win_len_s=win_len_s,
        hop_len_s=hop_len_s,
    )

    # Force fixed-length output (this is what prevents your vstack error)
    return summarize_frames_fixed(plp_feat, target_dim=n_ceps)


# ----------------------------
# Model + experiment runner
# ----------------------------

@dataclass
class ExperimentConfig:
    sample_rate: int = 16000
    test_size: float = 0.2
    random_state: int = 42
    classifier: str = "knn"  # "knn" or "linear"
    n_neighbors: int = 7


def make_model(cfg: ExperimentConfig):
    if cfg.classifier == "knn":
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("clf", KNeighborsClassifier(n_neighbors=cfg.n_neighbors)),
            ]
        )
    if cfg.classifier == "linear":
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("clf", LinearSVC(C=1.0)),
            ]
        )
    raise ValueError("classifier must be 'knn' or 'linear'")


def evaluate_feature_set(
    name: str,
    train_items: List[Tuple[Path, int]],
    test_items: List[Tuple[Path, int]],
    cfg: ExperimentConfig,
    snr_list: List[float],
    feature_extractor: Callable[[Path, int, Optional[float], np.random.Generator], np.ndarray],
) -> None:
    print("\n" + "=" * 50)
    print(f"Feature set : {name}")
    print(f"Classifier  : {cfg.classifier}")
    print(f"Train/Test  : {len(train_items)} / {len(test_items)}")
    print("=" * 50)

    rng_train = np.random.default_rng(cfg.random_state)
    rng_test_clean = np.random.default_rng(cfg.random_state + 1)

    X_train, y_train = [], []
    for p, lab in tqdm(train_items, desc=f"[{name}] Extract TRAIN (clean)"):
        X_train.append(feature_extractor(p, cfg.sample_rate, None, rng_train))
        y_train.append(lab)
    X_train = np.vstack(X_train)
    y_train = np.asarray(y_train, dtype=np.int64)

    X_test_clean, y_test = [], []
    for p, lab in tqdm(test_items, desc=f"[{name}] Extract TEST (clean)"):
        X_test_clean.append(feature_extractor(p, cfg.sample_rate, None, rng_test_clean))
        y_test.append(lab)
    X_test_clean = np.vstack(X_test_clean)
    y_test = np.asarray(y_test, dtype=np.int64)

    model = make_model(cfg)
    model.fit(X_train, y_train)

    y_pred_clean = model.predict(X_test_clean)
    acc_clean = accuracy_score(y_test, y_pred_clean)
    print(f"\n[{name}] CLEAN accuracy: {acc_clean:.4f}")
    print("Confusion matrix (clean):")
    print(confusion_matrix(y_test, y_pred_clean))

    for snr in snr_list:
        rng_test_noisy = np.random.default_rng(cfg.random_state + 1000 + int(round(snr * 10)))
        X_test_noisy = []
        for p, _lab in tqdm(test_items, desc=f"[{name}] Extract TEST (noisy, SNR={snr}dB)"):
            X_test_noisy.append(feature_extractor(p, cfg.sample_rate, snr, rng_test_noisy))
        X_test_noisy = np.vstack(X_test_noisy)

        y_pred_noisy = model.predict(X_test_noisy)
        acc_noisy = accuracy_score(y_test, y_pred_noisy)

        print(f"\n[{name}] NOISY accuracy @ SNR={snr} dB: {acc_noisy:.4f}")
        print("Confusion matrix (noisy):")
        print(confusion_matrix(y_test, y_pred_noisy))


# ----------------------------
# Argparse (Notebook-safe)
# ----------------------------

def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, default="./fsdd_data", help="Where to store/download FSDD")
    parser.add_argument("--sr", type=int, default=16000, help="Resample audio to this sampling rate")
    parser.add_argument("--test_size", type=float, default=0.2, help="Test split fraction")
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    parser.add_argument("--classifier", type=str, default="knn", choices=["knn", "linear"])
    parser.add_argument("--n_neighbors", type=int, default=7, help="K for KNN (if classifier=knn)")
    parser.add_argument("--snr_list", type=float, nargs="+", default=[10.0], help="SNR dB values for noisy test")
    return parser


def parse_args_notebook_safe(parser: argparse.ArgumentParser, argv: Optional[List[str]]):
    if argv is not None:
        return parser.parse_args(argv)
    args, _unknown = parser.parse_known_args()
    return args


def main(argv: Optional[List[str]] = None):
    parser = build_parser()
    args = parse_args_notebook_safe(parser, argv)

    cfg = ExperimentConfig(
        sample_rate=args.sr,
        test_size=args.test_size,
        random_state=args.seed,
        classifier=args.classifier,
        n_neighbors=args.n_neighbors,
    )

    data_root = Path(args.data_dir)
    recordings_dir = download_and_prepare_fsdd(data_root)
    items = list_dataset(recordings_dir)

    labels = np.array([y for (_p, y) in items], dtype=np.int64)
    idx = np.arange(len(items))

    train_idx, test_idx = train_test_split(
        idx,
        test_size=cfg.test_size,
        random_state=cfg.random_state,
        stratify=labels,
    )

    train_items = [items[i] for i in train_idx]
    test_items = [items[i] for i in test_idx]

    evaluate_feature_set(
        name="MFCC",
        train_items=train_items,
        test_items=test_items,
        cfg=cfg,
        snr_list=list(args.snr_list),
        feature_extractor=extract_mfcc_vector,
    )

    evaluate_feature_set(
        name="PLP",
        train_items=train_items,
        test_items=test_items,
        cfg=cfg,
        snr_list=list(args.snr_list),
        feature_extractor=extract_plp_vector,
    )

    print("\nDone. Compare MFCC vs PLP accuracies in CLEAN vs NOISY conditions.")


if __name__ == "__main__":
    main(["--snr_list", "20", "10", "0", "--classifier", "knn"])


Feature set : MFCC
Classifier  : knn
Train/Test  : 2400 / 600


[MFCC] Extract TEST (clean): 100%|██████████| 600/600 [00:02<00:00, 204.65it/s]



[MFCC] CLEAN accuracy: 0.9783
Confusion matrix (clean):
[[57  0  2  1  0  0  0  0  0  0]
 [ 0 59  0  0  0  0  0  0  0  1]
 [ 1  0 59  0  0  0  0  0  0  0]
 [ 1  0  1 57  0  0  0  0  1  0]
 [ 0  0  0  0 60  0  0  0  0  0]
 [ 0  0  0  1  0 59  0  0  0  0]
 [ 0  0  0  1  0  0 58  0  1  0]
 [ 0  0  0  0  0  0  0 60  0  0]
 [ 0  0  0  1  0  0  0  0 59  0]
 [ 0  0  0  0  0  0  0  1  0 59]]


[MFCC] Extract TEST (noisy, SNR=20.0dB): 100%|██████████| 600/600 [00:04<00:00, 132.90it/s]



[MFCC] NOISY accuracy @ SNR=20.0 dB: 0.1633
Confusion matrix (noisy):
[[ 4  0  0  7  0  0 49  0  0  0]
 [ 0  3  0  0  0  0 56  1  0  0]
 [ 0  0 11  1  0  0 48  0  0  0]
 [ 3  0  0  8  0  0 49  0  0  0]
 [ 0  0  0  0 13  0 47  0  0  0]
 [ 0  0  0  0  0  0 59  1  0  0]
 [ 0  0  0  1  0  0 59  0  0  0]
 [ 0  0  0  0  0  0 60  0  0  0]
 [ 0  0  0  0  0  0 60  0  0  0]
 [ 0  0  0  0  0  0 57  3  0  0]]


...


[PLP] NOISY accuracy @ SNR=20.0 dB: 0.1033
Confusion matrix (noisy):
[[ 0 17  3  3  0  0  0  1  0 36]
 [ 0 20  3 13  0  0  0  0  0 24]
 [ 4 10  7  6  0  0  0  0  0 33]
 [ 0 18  3  8  4  4  0  4  0 19]
 [ 0  8  0  8  0  0  0  0  0 44]
 [ 1 27  0 11  1  9  0  0  0 11]
 [ 6 15  7  5 10 13  0  1  0  3]
 [ 3 27  1  5  0  2  0  1  0 21]
 [ 0 31  4  2  1 13  0  2  0  7]
 [ 0 25  1 13  1  3  0  0  0 17]]


[PLP] Extract TEST (noisy, SNR=10.0dB): 100%|██████████| 600/600 [00:17<00:00, 33.44it/s]



[PLP] NOISY accuracy @ SNR=10.0 dB: 0.1050
Confusion matrix (noisy):
[[ 0 12  0 30  0  0  0  0  0 18]
 [ 0  8  0 11  0  2  0  0  0 39]
 [ 0 13  1 32  0  2  0  0  0 12]
 [ 0 11  0 28  5 10  0  0  0  6]
 [ 0  7  7 14  0  0  0  0  0 32]
 [ 0 11  1 16  0  7  0  0  0 25]
 [ 1 16  0 10  5 18  0  2  0  8]
 [ 0 16  0 16  2  8  0  0  0 18]
 [ 0 18  0 13  2 22  0  0  0  5]
 [ 0 11  0 23  2  5  0  0  0 19]]


[PLP] Extract TEST (noisy, SNR=0.0dB): 100%|██████████| 600/600 [00:19<00:00, 31.43it/s]


[PLP] NOISY accuracy @ SNR=0.0 dB: 0.0717
Confusion matrix (noisy):
[[ 0  0  2  0  1 57  0  0  0  0]
 [ 0  9  0  0  1 50  0  0  0  0]
 [ 0  1  3  0  3 53  0  0  0  0]
 [ 0  0  2  0 10 48  0  0  0  0]
 [ 0  3  1  0  0 56  0  0  0  0]
 [ 0 26  0  3  0 31  0  0  0  0]
 [ 0  3  1  0  7 49  0  0  0  0]
 [ 0 12  0  0  1 47  0  0  0  0]
 [ 0  2  0  0  5 53  0  0  0  0]
 [ 0  9  0  0  0 51  0  0  0  0]]

Done. Compare MFCC vs PLP accuracies in CLEAN vs NOISY conditions.


In the clean condition, **MFCC + KNN** achieves very high recognition accuracy (**97.83%**), with a near-diagonal confusion matrix, indicating that the spectral envelope information captured by MFCCs is highly discriminative for FSDD when there is no corruption. In contrast, **PLP + KNN** performs much worse even in clean speech (**47.83%**), suggesting that in this implementation/setup the PLP features are either less stable/discriminative (e.g., due to how the PLP coefficients are computed/normalized in the library version used) or that the simple “mean+std” utterance-level summarization and KNN classifier are not a good match for the PLP representation. Under noisy test conditions, performance collapses for both feature sets: MFCC drops sharply to **16.33% at 20 dB** and approaches chance level (~10%) at **10 dB (11.00%)** and **0 dB (10.17%)**, with the model heavily biasing predictions toward a single digit (most often **“6”**), which indicates a strong train–test mismatch because the classifier was trained only on clean data and the global statistics of the features shift significantly when noise is added. PLP does not show the expected robustness advantage here either (≈**10.33%**, **10.50%**, **7.17%** for 20/10/0 dB), so overall these results imply that, for this pipeline, **MFCC is clearly superior in clean speech**, while **neither MFCC nor PLP is robust to additive noise without noise-aware training, normalization, or more robust modeling** (e.g., adding noisy training data, using CMVN, delta features, or a model that handles temporal structure).
